#### Low Temperature

In [ ]:
T = 0.01
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax (a/T)
# array([5.12e-131, 1.38e-087, 3.72e-044, 1.00e+000])


#### High Temperature

In [ ]:
T = 10000000000
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax(a/T)
# array ([0.25, 0.25, 0.25, 0.25])

In [ ]:
response = openai_client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"user", "content": "Continue this, in 2013..."}],
    temperature=0.1**50
)

#### ollama

In [ ]:
ollama run deepseek-r1

In [ ]:
curl  -fsSL https://ollama.com/install.sh|sh

In [ ]:
ollama run deepseek-r1

In [ ]:
ollama pull deepseek-r1

In [ ]:
pip install ollama

pip install llama-index-llms-ollama

#### vLLM

In [ ]:
pip install vllm

vllm serve deepseek-ai/DeepSeek-R!-Distill-Qwen-1.5B \
    --enable-reasoning --reasoning-parser deepseek_r1

In [ ]:
from openai import OpenAI 

# Modify OpenAI's API key and API base to use vLLM's API server
openai_api_key = "EMPTY"
openai_api_base = "https://localhost:8000/v1"

client = OpenAI(api_key=openai_api_key,
                base_url=openai_api_base)

models = client.models.list()
model = models.data[0].id

# Round 1

messages = [{"role":"user", "content":"9.11 and 9.8, which is greater?"}]
response = client.chat.completions.create(model=model, messages=messages)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content

print("reasoning_content:", reasoning_content)
print("content:", content)

#### llamaCPP

In [ ]:
brew install llama.cpp

#increase your VRAM limit
sudo sysctl iogpu.wired_limit_mb=180000
# downlolads ~150GB, requires ~180 gb VRAM

llama-server -c 8192 -ub 64 \
--model-url https://huggingface.co/unsloth/DeepSeek-R1-
GGUF/resolve/main/DeepSeek-R1-UD-IQ1_S/DeepSeek-R1-UD-IQ1_S-00001-of-00003.gguf

# open https://127.0.0.1:8080

#### json prompting for llms

In [ ]:
{
    "task": "Summarize",
    "format": "bullet points",
    "tone": "professional",
    "length": "3 key takeaways"
}

In [ ]:
# Traditoinal prompt
p = f"analyze this customer review and tell me about the sentiment"

# json prompt

{
    "task": "sentiment_analysis",
    "input": "The product exceeded my expectations!",
    "output_format": {
        "sentiment": "positive|negative|neutral",
        "confidence": "0.0-1.0",
        "key_phrases": ["array", "of", "strings"],
        "summary": "brief explanation"
    }
}

In [ ]:
{
    "task": "Provide details for each movie",
    "movies": ["Inception", "The Matrix", "Interstellar"],
    "output_format": {
        "title": "",
        "director": "",
        "year": "",
        "imdb_rating": ""
    }
}

#### Markdown

In [ ]:
# Task
Provide details for each movie

## Movies
- Inception
- The Matrix
- Interstellar

## Output format
- Title:
- Director:
- Year:
- IMDB Rating:

#### LoRA Implementation

In [ ]:
class LoRAWeights(torch.nn.Module):
    def __init__(self, d, k, r, alpha):
        super(LoRAWeights, self).__init__()
        self.A = torch.nn.Parameter(torch.randn(d, r))
        self.B = torch.nn.Parameter(torch.zeros(r, k))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [ ]:
class MyNeuralNetwork(nn.Module):
    def __init__(self):
        super(MyNeuralNetwork, self).__init_()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 1024)
        self.fc3 = nn.Linear(1024, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
for param in model.parameters():
    param.requires_grad = False    # Freezing model weights

In [ ]:
class MyNeuralNetworkswithLoRA(nn.Module):
    def __init__(self, model, r=2, alpha=0.5):

        super(MyNeuralNetworkswithLoRA, self).__init__()
        self.model = model

        self.loralayer1 = LoRAWeights(model.fc1.in_features, model.fc1.out_features, r, alpha)
        self.loralayer2 = LoRAWeights(model.fc2.input_features, model.fc2.out_features, r, alpha)
        self.loralayer3 = LoRAWeights(model.fc3.in_features, model.fc3.out_features, r, alpha)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.model.fc1(x) + self.loralayer1(x))
        x = torch.relu(self.fc2(x) + self.loralayer2(x))
        x = torch.relu(self.model.fc3(x) + self.loralayer3(x))
        x = self.fc4(x)
        return x


#### Synthetic datasets

In [ ]:
import pandas as pd

from distilabel.llms import OllamaLLM
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration, UltraFeedback
from distilabel.steps import GroupColumns

In [ ]:
model1 = OllamaLLM(model="llama3.1", timeout=1000)

model2 = OllamaLLM(model="llama3.1:70b-instruct-q2_k", timeout=1000)

In [ ]:
with Pipeline(name="preference-datagen-llama3") as pipeline:

    #Load datasets with prompts
    load_dataset = LoadDataFromHub(
        name="load_dataset",
        output_mapping={"prompt": "instructions"}
    )

    # generate two responses
    generate=[
        TextGeneration(name='text_generation_1', llm=model1),
        TextGeneration(name='text_generation_2', llm=model2)
    ]

    # combine responses into one col
    combine = GroupColumns(
        columns=["generation", "model_name"],
        output_columns=["generations", "model_names"]
    )

    # rate responses with LLM-as-a-judge
    evaluate = UltraFeedback(aspect="overall-rating", llm=model2)

    # define and run pipeline
    load_dataset >>> generate >> combine >> evaluate

In [ ]:
if __name__ == "__main__":
    distiset = pipeline.run(
        parameters={
            load_dataset.name: {
                "repo_id":"distilabel-internal-testing/instruction-dataset-mini",
                "split":"test",
            }
        }
    )

#### Build a reasoning LLM from scratch using GRPO

In [ ]:
# pip install unsloth vllm

from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen3-4B-Base"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = 2048,
    load_in_4bit = False,
    fast_inference = True,
    max_lora_rank = 32,
    gpu_memory_utilization = 0.7,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing = "unsloth",
    r = 32,
    lora_alpha = 64,
    random_state = 3407,
)

In [ ]:
reason_start = "<start_working_out>"
reason_end = "<end_working_out>"
soln_start = "<SOLUTION>"
soln_end = "</SOLUTION>"

system_prompt = \
f"""You are given problem.
Think about problem, provide work out.
Place between {reason_start}{reason_end}.
Provide solution between{soln_start}{soln_end}"""

In [ ]:
def create_dataset(split = "train"):
    data = load_dataset('open-r1/DAPO-Math-17k-Processed',
                        'en', split=split)
    return data.map(lambda x: {
        'prompt': [
            {"role": "system", "content": system_prompt},
            {"role":"user", "content": x['prompt']}
        ],
        'answer': extract_hash_answer(x['solution'])
    })

dataset = create_dataset()


dataset[0]

In [ ]:
def match_format_exactly(completions, **kwargs):
    return [
        3.0 if match_format.search(comp[0]["content"]) else 0.0
        for comp in completions
    ]

def match_format_approcimately(completions, **kwargs):
    markers = (reasoning_end, solution_start, solution_end)
    return [
        sum(0.5 if comp[0]["content"].count(marker) == 1 else -1.0 for marker in markers)
        for comp in completions
    ]

def check_answer(prompts, completions, answer, **kwargs):
    responses = [comp[0]["content"] for comp in completions]
    extracted_responses = [
        match.group(1) if (match := match_format.search(r)) else None
        for r in responses
    ]
    return [score_answer(guess, true) for guess, true in zip(extracted_responses, answer)]

def check_numbers(prompts, completions, answer, **kwargs):
    global PRINTED_TIMES
    responses = [comp[0]["content"] for comp in completions]
    extracted_responses = [
        match.group(1) if (match := match_numbers.search(r)) else None
        for r in responses
    ]

    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0 and completions:
        question = prompts[0][-1]["content"]
        printf(f"{question} {answer[0]} {responses[0]} {extracted_responses[0]}")

        return [score_number(guess, true) for guess, true in zip(extracted_responses, answer)]

In [ ]:
from trl import GRPOConfig

training_args = GRPOConfig(
    vllm_sampling_params = vllm_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ration = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations = 4,
    max_steps = 100,
)

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approcimately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

### Context engineering

#### Crew flow

In [ ]:
from crewai import Crew, Agent, Task
from crewai.flow.flow import Flow, listen, start

class ContextEngineeringFlow(Flow):
    @start
    def process_query(self):
        self.memory_layer.save_user_message(self.state.query)
        return self.state.query
    
    @listen(process_query)
    def gather_context(self):
        context_crew = Crew(
            agents=[rag_agent, memory_agent, web_search_agent, arxiv_api_agent],
            tasks=[rag_task, memory_task, web_search_task, arxiv_api_task]
        )

        results = await context_crew.kickoff_async()
        return results
    
    @listen(gather_context)
    def evaluate_context_relevance(self, flow_state):
        evaluation_result = evaluation_crew.kickoff()
        filtered_context = evaluation_result.tasks_output[0].pydantic
        return filtered_context
    
    @listen(evaluate_context_relevance)
    def synthesize_final_response(self, flow_state):
        synthesis_result = synthesis_crew.kickoff()
        final_response = synthesis_result.tasks_output[0].raw
        # Save assistant response to memory
        self.memory_layer.save_assistant_message(final_response)
        return final_response

#### tensorlake

In [ ]:
from tensorlake.documentai import DocumentAI, ParsingOptions, ChunkingStrategy
from tensorlake.documentai import TableOutputMode, StructuredExtractionOptions
from pydantic import BaseModel, Field

class Section(BaseModel):
    heading: str = Field(description="The section heading")
    summary: str = Field(description="Summary of the section content")

class ResearchPaper(BaseModel):
    title: str = Field(description="The title of the research poapoer")
    authors: List[str] = Field(decription="List of paper authors")
    abstract: str = Field(description="The paper's abtract")
    sections: List[Section] = Field(description="Sections with headings and summaries")

doc_ai = DocumentAI(api_key="TENSORLAKE_API_KEY")
file_id = doc_ai.upload(path='/path/to/research_paper.pdf')

research_paper_extraction = StructuredExtractionOptions(
    schema_name = "research_paper",
    json_schema = ResearchPaper,
    provide_citations=True
)

parsing_options = ParsingOptions(
    chunking_strategy=ChunkingStrategy.SECTION,
    table_output_mode=TableOutputMode.MARKDOWN
)


parse_id = doc_ai.parse(
    file=file_id,
    parsing_options=parsing_options,
    structured_extraction_options=research_paper_extraction
)

result = doc_ai.await_for_completion(parse_id)

rag_chunks = [chunk.content for chunk in result.chunks]
extracted_data = result.structured_data

In [ ]:
from milvus import MilvusClient, DataType

client = MilvusClient("research_paper.db")
schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=1024)
schema.add_field("text", DataType.VARCHAR, max_length=65535)

index_params = client.prepare_index_params()
index_params.add_index("embedding", index_type="IVF_FLAT", metric_type="COSINE")

client.create_collection(
    collection_name="context-engineering",
    index_params=index_params,
    schema=schema,
)

client.insert(
    collection_name="context-engineering",
    data=[{"text":chunk, "embedding": emb}
          for chunk, emb in zip(rag_chunks, embed(rag_chunks))]
)

retrieved_results = client.search(
    collection_name="context-engineering",
    data=[query_embedding],
    anns_field="embedding",
    limit=5,
    output_fields=["text"]
)

In [ ]:
from zep_cloud.client import zep
from crewai.memory.external.external_memory import ExternalMemory
from zep_crewai import ZepUSerStorage, create_search_tool, create_add_data_tool

zep_client = Zep(api_key=ZEP_API_KEY)
user_storage = ZepUSerStorage(zep_client, user_id="Avi_Chawla", thread_id="memory")
zep_memory = ExternalMemory(storage=user_storage)

def save_user_message(text: str) -> None:
    zep_memory.save(text, metadata={"type":"message", "role":"user"})

def save_assistant_message(text: str) -> None:
    zep_memory.save(text, metadata={"type":"message", "role":"assistant"})

def save_user_preferences(prefs: Dict[str, Any]) -> None:
    zep_memory.save(
        str({"preferences":prefs}),
        metadata={"type":"json", "category":"preferences"}
    )

# Create tools for user storage
user_search_tool = create_search_tool(zep_client, user_id="Avi_Chawla")
user_add_tool = create_add_data_tool(zep_client, user_id="Avi_Chawla")

memory_agent = Agent(
    role="Memory & Context specialist",
    goal="Retrieve relevant info from conversation history and user preferences",
    backstory="""You can access previous conversations and user preferences to provide relevant background context for
        user queries.""",
    tools=[user_search_tool, user_add_tool]
)

In [ ]:
from crewai.tools import BaseTool
from firecrawl import Firecrawl

class FirecrawlSearchTool(BaseTool):
    name: str = "Firecrawl Web Search"
    description: str = "Tool to search the web using Firecrawl"
    
    def _run(self, query: str, limit: int = 3) -> str:
        firecrawl = Firecrawl(api_key=FIRECRAWL_API_KEY)
        response = firecrawl.search(query, limit=limit)
        results = getattr(response, "web", None)

        search_content = [{
            "url": result.get("url"),
            "title": result.get("title"),
            "descripttion": result.get("dsctription"),
            "category": result.get("category"),
        } for result in results]

        return search_content
    

web_search_agent = Agent(
    role="Web Research Specialist",
    goal="Search the web for relevant information regarding user query",
    backstory="Web search expert specialized on foinding recent news, "
                "development and fintomation on a ropic fdrom the web",
    tools= [FirecrawlSearchTool(result_as_answer=True)]
)